In [ ]:
!pip install mp-api pymatgen
!pip install shap
import shap
from mp_api.client import MPRester
import pandas as pd
import time
from google.colab import files

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
df=pd.read_csv("topological_elements_100_sampled.csv")
print(df.head())
print(df.columns)
print(df.shape)

#creating new column 'label'
df['Label']=df['Classification'].apply(lambda x: 0 if str(x).lower()=='trivial' else 1)
df = df[["Compound","Classification","SubClassification","Label"]]
print(df.head())

In [ ]:
df=df.drop_duplicates(subset=['Compound'])
print(df.shape)

materials = df["Compound"].tolist()
print(materials[:20])
print("Total materials:", len(materials))

In [ ]:
API_KEY = "dm85eSraFBImtIN0QNikcr0v70eZb3WQ"

sample_df = df.sample(n=3500, random_state=42)
materials = sample_df["Compound"].tolist()

materials_data = []

with MPRester(API_KEY) as mpr:
    for formula in materials:
        try:
            docs = mpr.materials.summary.search(
                formula=[formula],
                fields=[
                    "material_id",
                    "formula_pretty",
                    "band_gap",
                    "density",
                    "formation_energy_per_atom",
                    "volume",
                    "nsites",
                    "symmetry"
                ]
            )

            if len(docs) > 0:
                doc = docs[0]

                materials_data.append({
                    "MaterialID": doc.material_id,
                    "Compound": formula,
                    "BandGap": doc.band_gap,
                    "Density": doc.density,
                    "FormationEnergy": doc.formation_energy_per_atom,
                    "Volume": doc.volume,
                    "Sites": doc.nsites,
                    "CrystalSystem": str(doc.symmetry.crystal_system) if doc.symmetry else None,
                    "SpaceGroup": str(doc.symmetry.symbol) if doc.symmetry else None
                })

            time.sleep(0.1)

        except Exception as e:
            print(f"Error for {formula}: {e}")

descriptor_df = pd.DataFrame(materials_data)

final_df = pd.merge(sample_df, descriptor_df, on="Compound")

print(final_df.shape)
print(final_df.head())

final_df.to_csv("topological_3500_with_descriptors.csv", index=False)

In [ ]:
files.download("topological_3500_with_descriptors.csv")

In [ ]:
final_df = pd.read_csv("topological_3500_with_descriptors.csv")
print(final_df.shape)
print(final_df.isnull().sum())
final_df = final_df.dropna()
print(final_df.shape)

In [ ]:
le_crystal = LabelEncoder()
le_space = LabelEncoder()

final_df["CrystalSystem"] = le_crystal.fit_transform(final_df["CrystalSystem"])
final_df["SpaceGroup"] = le_space.fit_transform(final_df["SpaceGroup"])

X = final_df[[
    "BandGap",
    "Density",
    "FormationEnergy",
    "Volume",
    "Sites",
    "CrystalSystem",
    "SpaceGroup"
]]

y = final_df["Label"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
lr = LogisticRegression(
    max_iter=3000,
    random_state=42
)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

svm = SVC(
    probability=True,
    kernel='rbf',
    random_state=42
)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)

dt = DecisionTreeClassifier(
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

results = pd.DataFrame({
    "Model": ["Logistic Regression", "SVM", "Decision Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, svm_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Precision": [
        precision_score(y_test, lr_pred),
        precision_score(y_test, svm_pred),
        precision_score(y_test, dt_pred),
        precision_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, lr_pred),
        recall_score(y_test, svm_pred),
        recall_score(y_test, dt_pred),
        recall_score(y_test, rf_pred)
    ],
    "F1 Score": [
        f1_score(y_test, lr_pred),
        f1_score(y_test, svm_pred),
        f1_score(y_test, dt_pred),
        f1_score(y_test, rf_pred)
    ]
})

print(results)

In [ ]:
lr_probs = lr.predict_proba(X_test)[:, 1]
svm_probs = svm.predict_proba(X_test)[:, 1]
dt_probs = dt.predict_proba(X_test)[:, 1]
rf_probs = rf.predict_proba(X_test)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_probs)
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)

auc_lr = auc(fpr_lr, tpr_lr)
auc_svm = auc(fpr_svm, tpr_svm)
auc_dt = auc(fpr_dt, tpr_dt)
auc_rf = auc(fpr_rf, tpr_rf)

plt.figure(figsize=(8,6))

plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {auc_lr:.2f})")
plt.plot(fpr_svm, tpr_svm, label=f"SVM (AUC = {auc_svm:.2f})")
plt.plot(fpr_dt, tpr_dt, label=f"Decision Tree (AUC = {auc_dt:.2f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {auc_rf:.2f})")

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison of ML Models")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.bar(results["Model"], results["Accuracy"])

for i, value in enumerate(results["Accuracy"]):
    plt.text(i, value + 0.005, f"{value:.3f}", ha='center')

plt.ylim(0.75, 0.95)
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.title("Accuracy Comparison of ML Models")
plt.xticks(rotation=20)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, rf_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix for Random Forest")
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, rf_probs)

plt.figure(figsize=(7,5))
plt.plot(recall, precision)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve for Random Forest")
plt.show()

In [ ]:
importance = rf.feature_importances_

feature_names = [
    "BandGap",
    "Density",
    "FormationEnergy",
    "Volume",
    "Sites",
    "CrystalSystem",
    "SpaceGroup"
]

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

importance_df = importance_df.sort_values(by="Importance", ascending=False)

print(importance_df)

plt.figure(figsize=(8,5))
plt.bar(importance_df["Feature"], importance_df["Importance"])

plt.xticks(rotation=45)
plt.xlabel("Descriptors")
plt.ylabel("Importance")
plt.title("Feature Importance in Random Forest")

plt.show()

In [ ]:
X_test_df = pd.DataFrame(X_test, columns=feature_names)
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test_df)
shap.summary_plot(shap_values[:, :, 1], X_test_df)

In [ ]:

def predict_material_by_formula(formula):
    with MPRester(API_KEY) as mpr:
        docs = mpr.materials.summary.search(
            formula=[formula],
            fields=[
                "band_gap",
                "density",
                "formation_energy_per_atom",
                "volume",
                "nsites",
                "symmetry"
            ]
        )

    if len(docs) == 0:
        print("Material not found in Materials Project database")
        return

    doc = docs[0]

    band_gap = doc.band_gap
    density = doc.density
    formation_energy = doc.formation_energy_per_atom
    volume = doc.volume
    sites = doc.nsites
    crystal_system = str(doc.symmetry.crystal_system) if doc.symmetry else None
    space_group = str(doc.symmetry.symbol) if doc.symmetry else None

    crystal_encoded = le_crystal.transform([crystal_system])[0]
    space_encoded = le_space.transform([space_group])[0]

    user_data = pd.DataFrame([{
        "BandGap": band_gap,
        "Density": density,
        "FormationEnergy": formation_energy,
        "Volume": volume,
        "Sites": sites,
        "CrystalSystem": crystal_encoded,
        "SpaceGroup": space_encoded
    }])

    user_data = scaler.transform(user_data)

    prediction = rf.predict(user_data)[0]

    print("Descriptors Found:")
    print("Band Gap:", band_gap)
    print("Density:", density)
    print("Formation Energy:", formation_energy)
    print("Volume:", volume)
    print("Sites:", sites)
    print("Crystal System:", crystal_system)
    print("Space Group:", space_group)

    if prediction == 1:
        print("The material is predicted to be Topological")
    else:
        print("The material is predicted to be Non-Topological")
formula = input("Enter chemical formula: ")
predict_material_by_formula(formula)

In [ ]:
predict_material_by_formula("Bi2Se3")